# 과제3
* Memory store로 Recommender system 만들기
1) content-base
2) user-base

In [36]:
import pandas as pd
import numpy as np
from ast import literal_eval
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore

## 1) Content-base Recommender

### prep

In [45]:
orig_df = pd.read_csv('./data/tmdb_5000_movies.csv')
df = orig_df.copy()

df.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [46]:
df.info()               # 데이터에 대한 대략적인 정보 보기

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [47]:
df.iloc[0]["keywords"]        # 어떤 데이터가 들어있는지 확인하기 위해 찍어봄. str데이터인걸 확인

'[{"id": 1463, "name": "culture clash"}, {"id": 2964, "name": "future"}, {"id": 3386, "name": "space war"}, {"id": 3388, "name": "space colony"}, {"id": 3679, "name": "society"}, {"id": 3801, "name": "space travel"}, {"id": 9685, "name": "futuristic"}, {"id": 9840, "name": "romance"}, {"id": 9882, "name": "space"}, {"id": 9951, "name": "alien"}, {"id": 10148, "name": "tribe"}, {"id": 10158, "name": "alien planet"}, {"id": 10987, "name": "cgi"}, {"id": 11399, "name": "marine"}, {"id": 13065, "name": "soldier"}, {"id": 14643, "name": "battle"}, {"id": 14720, "name": "love affair"}, {"id": 165431, "name": "anti war"}, {"id": 193554, "name": "power relations"}, {"id": 206690, "name": "mind and soul"}, {"id": 209714, "name": "3d"}]'

In [48]:
from ast import literal_eval

result = literal_eval(df.iloc[0]["keywords"])     # str인 다른 데이터를 바로 사용할 수 있는 원래 데이터로 바꿔줌.
result

[{'id': 1463, 'name': 'culture clash'},
 {'id': 2964, 'name': 'future'},
 {'id': 3386, 'name': 'space war'},
 {'id': 3388, 'name': 'space colony'},
 {'id': 3679, 'name': 'society'},
 {'id': 3801, 'name': 'space travel'},
 {'id': 9685, 'name': 'futuristic'},
 {'id': 9840, 'name': 'romance'},
 {'id': 9882, 'name': 'space'},
 {'id': 9951, 'name': 'alien'},
 {'id': 10148, 'name': 'tribe'},
 {'id': 10158, 'name': 'alien planet'},
 {'id': 10987, 'name': 'cgi'},
 {'id': 11399, 'name': 'marine'},
 {'id': 13065, 'name': 'soldier'},
 {'id': 14643, 'name': 'battle'},
 {'id': 14720, 'name': 'love affair'},
 {'id': 165431, 'name': 'anti war'},
 {'id': 193554, 'name': 'power relations'},
 {'id': 206690, 'name': 'mind and soul'},
 {'id': 209714, 'name': '3d'}]

In [49]:
df["keywords"] = orig_df["keywords"].apply(literal_eval)  # 이렇게 해야 pandas 내부 함수이기 때문에 for문보다 훨씬 빠름

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [50]:
# 구조 파악하기
data_list = []

for i in df.iloc[0,:]['keywords']:
    data_list.append(i['name'])

data_list

['culture clash',
 'future',
 'space war',
 'space colony',
 'society',
 'space travel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alien planet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'love affair',
 'anti war',
 'power relations',
 'mind and soul',
 '3d']

In [51]:
# 장르 리스트화
df['keywords'] = df['keywords'].apply(
    lambda lst: [x['name'] for x in lst]
)

df['keywords'].head()

0    [culture clash, future, space war, space colon...
1    [ocean, drug abuse, exotic island, east india ...
2    [spy, based on novel, secret agent, sequel, mi...
3    [dc comics, crime fighter, terrorist, secret i...
4    [based on novel, mars, medallion, space travel...
Name: keywords, dtype: object

In [53]:
# SF와 같은 띄어쓰기로 되어있는 데이터 붙이기.
df["keywords"] = df["keywords"].apply(
    lambda genre_list: [''.join(g.split()) for g in genre_list]
)

df["keywords"].head()

0    [cultureclash, future, spacewar, spacecolony, ...
1    [ocean, drugabuse, exoticisland, eastindiatrad...
2    [spy, basedonnovel, secretagent, sequel, mi6, ...
3    [dccomics, crimefighter, terrorist, secretiden...
4    [basedonnovel, mars, medallion, spacetravel, p...
Name: keywords, dtype: object

In [54]:
# CountVectorizer에 넣을 수 있게 prep하기.
df["keywords"] = df["keywords"].apply(
    lambda genre_list: ' '.join(genre_list)
)

df["keywords"].head()

0    cultureclash future spacewar spacecolony socie...
1    ocean drugabuse exoticisland eastindiatradingc...
2    spy basedonnovel secretagent sequel mi6 britis...
3    dccomics crimefighter terrorist secretidentity...
4    basedonnovel mars medallion spacetravel prince...
Name: keywords, dtype: str

In [55]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer 모델 준비 - 단어사전을 만들고 등장한 단어 개수를 counting해준다.
count_vec = CountVectorizer(
    min_df=1, ngram_range=(1,1)
)

# 학습하여 변환 적용
genre_mat = count_vec.fit_transform(df["keywords"])

In [56]:
# 벡터라이저의 단어사전 확인하기 - 인덱스의 순서는 단어의 alphabetical order. 알파벳순
count_vec.vocabulary_

{'cultureclash': 2142,
 'future': 3459,
 'spacewar': 8197,
 'spacecolony': 8181,
 'society': 8112,
 'spacetravel': 8196,
 'futuristic': 3463,
 'romance': 7396,
 'space': 8176,
 'alien': 153,
 'tribe': 9084,
 'alienplanet': 166,
 'cgi': 1484,
 'marine': 5262,
 'soldier': 8127,
 'battle': 715,
 'loveaffair': 5096,
 'antiwar': 321,
 'powerrelations': 6766,
 'mindandsoul': 5514,
 '3d': 21,
 'ocean': 6115,
 'drugabuse': 2594,
 'exoticisland': 2927,
 'eastindiatradingcompany': 2677,
 'loveofone': 5106,
 'slife': 8028,
 'traitor': 9026,
 'shipwreck': 7865,
 'strongwoman': 8456,
 'ship': 7862,
 'alliance': 181,
 'calypso': 1285,
 'afterlife': 95,
 'fighter': 3201,
 'pirate': 6554,
 'swashbuckler': 8604,
 'aftercreditsstinger': 94,
 'spy': 8283,
 'basedonnovel': 677,
 'secretagent': 7685,
 'sequel': 7754,
 'mi6': 5455,
 'britishsecretservice': 1123,
 'unitedkingdom': 9238,
 'dccomics': 2236,
 'crimefighter': 2066,
 'terrorist': 8804,
 'secretidentity': 7694,
 'burglar': 1212,
 'hostagedrama': 4

In [57]:
# 넘파이 배열로 표현하기 - 등장한 단어 개수 카운팅
genre_mat.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4803, 9789))

In [58]:
# 배열을 데이터프레임으로 만들기, 
df_genre_vec = pd.DataFrame(genre_mat.toarray())
df_genre_vec.columns = sorted(count_vec.vocabulary_, key=count_vec.vocabulary_.get) # 단어의 순서대로 열에 적용한다.
df_genre_vec.index = df["title"]
df_genre_vec.head()

,11,15thcentury,16thcentury,17thcentury,18thcentury,1910s,1920s,1930s,1940s,1950s,...,zombie,zombieapocalypse,zombification,zoo,zookeeper,zurich,γη,卧底肥妈,绝地奶霸,超级妈妈
title,,,,,,,,,,,,,,,,,,,,,
Avatar,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Pirates of the Caribbean: At World's End,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Spectre,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
The Dark Knight Rises,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
John Carter,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [59]:
from sklearn.metrics.pairwise import cosine_similarity

genre_sim = cosine_similarity(df_genre_vec, df_genre_vec)   # 코사인 유사도로 비교한다.
genre_sim

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(4803, 4803))

In [60]:
genre_sim_df = pd.DataFrame(genre_sim)      # 유사도 표를 데이터프레임으로 만들어서 볼 수 있게 한다.
genre_sim_df.columns = df["title"]          # 영화 제목이 행과 열이기 때문
genre_sim_df.index = df["title"]
genre_sim_df

title,Avatar,Pirates of the Caribbean: At World's End,Spectre,The Dark Knight Rises,John Carter,Spider-Man 3,Tangled,Avengers: Age of Ultron,Harry Potter and the Half-Blood Prince,Batman v Superman: Dawn of Justice,...,On The Downlow,Sanctuary: Quite a Conundrum,Bang,Primer,Cavite,El Mariachi,Newlyweds,"Signed, Sealed, Delivered",Shanghai Calling,My Date with Drew
title,,,,,,,,,,,,,,,,,,,,,
Avatar,1.000000,0.0,0.000000,0.0,0.163663,0.000000,0.00000,0.072739,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Pirates of the Caribbean: At World's End,0.000000,1.0,0.000000,0.0,0.000000,0.117647,0.00000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Spectre,0.000000,0.0,1.000000,0.0,0.094491,0.091670,0.00000,0.125988,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
The Dark Knight Rises,0.000000,0.0,0.000000,1.0,0.000000,0.051709,0.00000,0.071067,0.0,0.213201,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
John Carter,0.163663,0.0,0.094491,0.0,1.000000,0.000000,0.06455,0.083333,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
El Mariachi,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Newlyweds,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"Signed, Sealed, Delivered",0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [61]:
genre_sim_df.iloc[0].sort_values(ascending=False) # 0번째 행에 있는 영화제목과 유사도값들을 내림차순으로 불러옴

title
Avatar                       1.000000
Star Trek Into Darkness      0.276026
A Monster in Paris           0.218218
Top Cat Begins               0.218218
Jupiter Ascending            0.195180
                               ...   
El Mariachi                  0.000000
Newlyweds                    0.000000
Signed, Sealed, Delivered    0.000000
Shanghai Calling             0.000000
Spectre                      0.000000
Name: Avatar, Length: 4803, dtype: float64

In [62]:
query = "Superman II"

index_no = df.loc[df["title"]==query,:].index[0]    # 쿼리를 만들어서 이런식으로 함수화가 가능함.

genre_sim_df.iloc[index_no].sort_values(ascending=False)

title
Superman II                  1.000000
Superman Returns             0.613941
Superman III                 0.585369
Superman                     0.470871
Man of Steel                 0.462250
                               ...   
El Mariachi                  0.000000
Newlyweds                    0.000000
Signed, Sealed, Delivered    0.000000
Shanghai Calling             0.000000
She Wore a Yellow Ribbon     0.000000
Name: Superman II, Length: 4803, dtype: float64